# Get Started with heliaPROFILER

heliaPROFILER (`hpx`) shows where time, processor events, and memory go when a LiteRT model runs on Ambiq Apollo hardware. It builds and flashes profiling firmware, captures layer-level PMU counters, and returns typed results you can inspect, compare, and export from Python.

This notebook introduces the interactive `hpx.Session` workflow and highlights the profiler's distinctive capabilities:

- check host tools and discover boards, probes, ports, engines, and PMU counters;
- branch one immutable experiment into heliaRT and heliaAOT configurations;
- analyze model operations before using hardware;
- capture and filter per-layer cycles and compute-unit counters;
- inspect memory placement, compare engines, and locate Model Explorer overlays;
- add optional Joulescope power measurement without changing the profiling workflow.

Hardware profiling, probe discovery, and power capture have separate controls. Keep `RUN_HARDWARE = False`, `RUN_PROBE_DISCOVERY = False`, and `RUN_POWER = False` for the safe introductory path. Enable only the operation and hardware you intend to use.

## 1. Choose Your Run Settings

Edit the board, transport, and hardware flags in the next cell. RTT is the recommended default and its target sources are bundled with HPX. GCC, CMake, Ninja, and J-Link should already be available to the notebook kernel; the readiness table below reports anything missing.

See the [transport guide](../../docs/guide/transports.md) for USB CDC, UART, SWO, and advanced RTT overrides.

In [1]:
from pathlib import Path
from statistics import mean, median
import csv
import json
import platform
import tempfile

from rich.console import Console
from rich.table import Table

import helia_profiler as hpx

console = Console(highlight=False)
MODEL = hpx.examples.tiny_cnn()
BOARD = "apollo510_evb"
TRANSPORT = "rtt"  # Alternatives: "usb_cdc", "uart", "swo"
RESULTS_ROOT = Path.cwd().resolve() / "results" / "notebook_showcase"
RT_RESULTS = RESULTS_ROOT / "helia_rt"
AOT_RESULTS = RESULTS_ROOT / "helia_aot"
COMPARE_RESULTS = RESULTS_ROOT / "rt_vs_aot"
EXPORT_DIR = RESULTS_ROOT / "exports"

RUN_HARDWARE = True
RUN_PROBE_DISCOVERY = True
RUN_POWER = False

print(f"hpx version:    {hpx.__version__}")
print(f"Python:         {platform.python_version()}")
print(f"Model:          {MODEL}")
print(f"Transport:      {TRANSPORT}")
print(f"Run hardware:   {RUN_HARDWARE}")
print(f"Discover probe: {RUN_PROBE_DISCOVERY}")
print(f"Capture power:  {RUN_POWER}")

hpx version:    0.1.0
Python:         3.11.15
Model:          /Users/adam.page/.cache/helia-profiler/models/tiny-cnn/4419dfff1e15/tiny_cnn.tflite
Transport:      rtt
Run hardware:   True
Discover probe: True
Capture power:  False


In [2]:
base = (
    hpx.Session()
    .with_model(MODEL)
    .with_target(board=BOARD, toolchain="gcc", transport=TRANSPORT)
    .with_profiling(
        iterations=100,
        warmup=5,
        pmu_counters={"cpu": "default", "memory": "default"},
    )
)

rt_session = (
    base
    .with_engine("helia-rt")
    .with_model(MODEL, arena_size=131_072)
    .with_output(dir=RT_RESULTS, detailed=True)
)
aot_session = (
    base
    .with_engine("helia-aot")
    .with_output(dir=AOT_RESULTS, detailed=True)
)

print("Base engine:", base.resolve().engine.type)
print("RT engine:  ", rt_session.resolve().engine.type)
print("AOT engine: ", aot_session.resolve().engine.type)
assert base.resolve().engine.type == hpx.EngineType.HELIA_RT

Base engine: helia-rt
RT engine:   helia-rt
AOT engine:  helia-aot


## 2. Profile the Packaged Example Model

`hpx.examples.tiny_cnn()` provides a deterministic six-operator int8 CNN with fixed batch size one:

`Conv2D 3×3 → AveragePool 2×2 → Conv2D 1×1 → Reshape → FullyConnected → Softmax`

The notebook therefore works from an installed HPX environment without a repository checkout or model download. Future packaged models and companion inputs will also live under `hpx.examples`. `Session.profile()` turns the model into a typed `ProfileResult` through the complete NSX configure, build, flash, capture, and report pipeline.

In [3]:
readiness = rt_session.show(rt_session.doctor())
rt_result = None

if RUN_HARDWARE and not readiness.ok:
    raise RuntimeError("Resolve the required environment checks before profiling")

if RUN_HARDWARE:
    rt_result = rt_session.profile()
    print(
        f"Loaded {rt_result.layer_count} layers, "
        f"{rt_result.total_cycles:,.0f} total cycles"
    )
else:
    print("Profile acquisition skipped. Set RUN_HARDWARE = True to build and flash.")

Environment Check                                                                                                  
╭────┬───────────────────────────────────────────┬───────────┬────────────────────────────────────────────────────╮
│    │ Dependency                                │ Status    │ Location                                           │
├────┼───────────────────────────────────────────┼───────────┼────────────────────────────────────────────────────┤
│ ✓  │ ARM GCC toolchain                         │ available │ /Applications/ArmGNUToolchain/15.2.rel1/arm-none-… │
│ ✓  │ CMake (>= 3.24)                           │ available │ /opt/homebrew/bin/cmake                            │
│ ✓  │ Ninja build system                        │ available │ /opt/homebrew/bin/ninja                            │
│ ✓  │ SEGGER J-Link commander                   │ available │ /usr/local/bin/JLinkExe                            │
│ ✓  │ neuralspotx Python package                │ available │                                                    │
│ ✓  │ pylink Python package (RTT/SWO transport) │ available │                                                    │
│ ✓  │ heliaAOT compiler                         │ available │                                                    │
│ ✓  │ ARM Compiler (armclang)                   │ available │                                                    │
│ ✓  │ ARM fromelf (armclang)                    │ available │                                                    │
│ ✓  │ SEGGER RTT source checkout                │ available │ /Users/adam.page/Ambiq/helia/helia-profiler/src/h… │
╰────┴───────────────────────────────────────────┴───────────┴────────────────────────────────────────────────────╯
                                         All required dependencies found.                                          

/Users/adam.page/Ambiq/helia/helia-profiler/.venv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Cloning into 
'/Users/adam.page/.cache/helia-profiler/workspaces/apollo510_evb-arm-none-eabi-gcc-helia-rt/profiler_app/modules/ns
x-ambiq-sdk'...

/Users/adam.page/Ambiq/helia/helia-profiler/.venv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/adam.page/Ambiq/helia/helia-profiler/.venv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

  Section       Size  
  text       252,388  
  data       103,544  
  bss        502,912  
  total      858,844

Memory Plan (helia-rt)                                                                                             
 Region         Used     Capacity                                 %   Consumers                                    
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 MRAM            0 B      4.00 MB   ╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌         0%   —                                            
 SRAM            0 B      3.00 MB   ╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌         0%   —                                            
 DTCM       130.4 KB     512.0 KB   ━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌        25%   model_flatbuffer=2.4 KB, tensor_arena=128.0  
                                                                      KB                                           
 ITCM            0 B     256.0 KB   ╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌         0%   —                                            
 PSRAM           0 B     64.00 MB   ╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌         0%   —                                            

Cache & Memory                             
 Counter                             Total 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 L1D_CACHE_REFILL                        0 
 MEM_ACCESS                         14,686 
 BUS_ACCESS                              8

╭─ Output → ]8;id=9427901;file:///Users/adam.page/Ambiq/helia/helia-profiler/examples/notebooks/results/notebook_showcase/helia_rt\/Users/adam.page/Ambiq/helia/helia-profiler/examples/notebooks/results/notebook_showcase/helia_rt]8;;\ ─╮
│    profile_results.csv                                                                                       │
│    summary.json                                                                                              │
│    run_metadata.json                                                                                         │
│    model_explorer/me_overlay_ARM_PMU_CPU_CYCLES.json                                                         │
│    model_explorer/me_overlay_ARM_PMU_INST_RETIRED.json                                                       │
│    model_explorer/me_overlay_ARM_PMU_STALL_FRONTEND.json                                                     │
│    model_explorer/me_overlay_ARM_PMU_STALL_BACKEND.json                                                      │
│    model_explorer/me_overlay_ARM_PMU_MEM_ACCESS.json                                                         │
│    model_explorer/me_overlay_ARM_PMU_L1D_CACHE_REFILL.json                                                   │
│    model_explorer/me_overlay_ARM_PMU_BUS_ACCESS.json                                                         │
│    model_explorer/me_overlay_ARM_PMU_BUS_CYCLES.json                                                         │
│    detailed/profile_cpu_0.csv                                                                                │
│    detailed/profile_memory_0.csv                                                                             │
│    detailed/profile_cpu.csv                                                                                  │
│    detailed/profile_memory.csv                                                                               │
│    detailed/memory.json                                                                                      │
│                                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────── 85.3s total ─╯

Loaded 6 layers, 39,341 total cycles


## 3. Inspect Profile Metadata and Available Metrics

Start with host readiness and discover the supported engines, boards, PMU groups, counters, and serial transports. Probe access is separately guarded because discovery communicates with SEGGER tools.

In [4]:
doctor = readiness
engines = base.show(base.engines())
boards = base.show(base.boards())
ports = base.show(base.ports())

print("Counter groups:", ", ".join(base.counter_groups()))
cpu_counters = base.show(base.counters("cpu"))

probes = ()
probe_matches = ()
if RUN_PROBE_DISCOVERY:
    probes = base.show(base.probes())
    probe_matches = base.show(base.inspect_probes())
else:
    print("Probe discovery skipped. Set RUN_PROBE_DISCOVERY = True to query J-Link.")

Inference    
Engines      
╭───────────╮
│ Engine    │
├───────────┤
│ tflm      │
│ helia-rt  │
│ helia-aot │
╰───────────╯

Boards                                             
╭────────────────────────┬──────────────┬─────────╮
│ Board                  │ SoC          │ Channel │
├────────────────────────┼──────────────┼─────────┤
│ apollo3p_evb           │ apollo3p     │ stable  │
│ apollo3p_evb_cygnus    │ apollo3p     │ preview │
│ apollo4p_evb           │ apollo4p     │ preview │
│ apollo4l_evb           │ apollo4l     │ preview │
│ apollo4l_blue_evb      │ apollo4l     │ preview │
│ apollo4p_blue_kbr_evb  │ apollo4p     │ preview │
│ apollo4p_blue_kxr_evb  │ apollo4p     │ preview │
│ apollo510_evb          │ apollo510    │ stable  │
│ apollo510b_evb         │ apollo510b   │ preview │
│ apollo5b_evb           │ apollo5b     │ preview │
│ apollo330mP_evb        │ apollo330P   │ preview │
╰────────────────────────┴──────────────┴─────────╯

Serial Ports                                                               
╭───────────────────────────────┬────────────┬──────────────┬─────────────╮
│ Device                        │ Kind       │ Serial       │ Description │
├───────────────────────────────┼────────────┼──────────────┼─────────────┤
│ /dev/cu.usbmodem0011600029541 │ jlink-vcom │ 001160002954 │ J-Link      │
╰───────────────────────────────┴────────────┴──────────────┴─────────────╯

Counter groups: cpu, memory, mve


PMU Counters                                                                                                       
╭────────────────────────────────┬───────┬───────┬────────────────────────────────────────────────────────────────╮
│ Counter                        │ Group │ Event │ Description                                                    │
├────────────────────────────────┼───────┼───────┼────────────────────────────────────────────────────────────────┤
│ ARM_PMU_SW_INCR                │ cpu   │ 0x00  │ Software update to the PMU_SWINC register, architecturally     │
│                                │       │       │ executed and condition code check pass                         │
│ ARM_PMU_L1I_CACHE_REFILL       │ cpu   │ 0x01  │ L1 I-Cache refill                                              │
│ ARM_PMU_LD_RETIRED             │ cpu   │ 0x06  │ Memory-reading instruction architecturally executed and        │
│                                │       │       │ condition code check pass                                      │
│ ARM_PMU_ST_RETIRED             │ cpu   │ 0x07  │ Memory-writing instruction architecturally executed and        │
│                                │       │       │ condition code check pass                                      │
│ ARM_PMU_INST_RETIRED           │ cpu   │ 0x08  │ Instruction architecturally executed                           │
│ ARM_PMU_EXC_TAKEN              │ cpu   │ 0x09  │ Exception entry                                                │
│ ARM_PMU_EXC_RETURN             │ cpu   │ 0x0a  │ Exception return instruction architecturally executed and the  │
│                                │       │       │ condition code check pass                                      │
│ ARM_PMU_PC_WRITE_RETIRED       │ cpu   │ 0x0c  │ Software change to the Program Counter (PC). Instruction is    │
│                                │       │       │ architecturally executed and condition code check pass         │
│ ARM_PMU_BR_IMMED_RETIRED       │ cpu   │ 0x0d  │ Immediate branch architecturally executed                      │
│ ARM_PMU_BR_RETURN_RETIRED      │ cpu   │ 0x0e  │ Function return instruction architecturally executed and the   │
│                                │       │       │ condition code check pass                                      │
│ ARM_PMU_UNALIGNED_LDST_RETIRED │ cpu   │ 0x0f  │ Unaligned memory memory-reading or memory-writing instruction  │
│                                │       │       │ architecturally executed and condition code check pass         │
│ ARM_PMU_CPU_CYCLES             │ cpu   │ 0x11  │ Cycle                                                          │
│ ARM_PMU_BR_RETIRED             │ cpu   │ 0x21  │ Branch instruction architecturally executed                    │
│ ARM_PMU_BR_MIS_PRED_RETIRED    │ cpu   │ 0x22  │ Mispredicted branch instruction architecturally executed       │
│ ARM_PMU_STALL_FRONTEND         │ cpu   │ 0x23  │ No operation issued because of the frontend                    │
│ ARM_PMU_STALL_BACKEND          │ cpu   │ 0x24  │ No operation issued because of the backend                     │
│ ARM_PMU_STALL                  │ cpu   │ 0x3c  │ Stall cycle for instruction or operation not sent for          │
│                                │       │       │ execution                                                      │
│ ARM_PMU_LE_RETIRED             │ cpu   │ 0x100 │ Loop end instruction executed                                  │
│ ARM_PMU_LE_CANCEL              │ cpu   │ 0x108 │ Loop end instruction not taken                                 │
│ ARM_PMU_SE_CALL_S              │ cpu   │ 0x114 │ Call to secure function, resulting in Security state change    │
│ ARM_PMU_SE_CALL_NS             │ cpu   │ 0x115 │ Call to non-secure function, resulting in Security state       │
│                                │       │       │ change                                                         │
╰────────────────────────────────┴───────┴───────┴──────

J-Link Probes                                            
╭──────────────┬───────────────────────────┬────────────╮
│ Serial       │ Product                   │ Connection │
├──────────────┼───────────────────────────┼────────────┤
│ 1160002954   │ J-Link-OB-Apollo4-CortexM │ USB        │
╰──────────────┴───────────────────────────┴────────────╯

J-Link Probe Targets                                        
╭──────────────┬───────────────────────────┬───────────────╮
│ Serial       │ Product                   │ Detected Core │
├──────────────┼───────────────────────────┼───────────────┤
│ 1160002954   │ J-Link-OB-Apollo4-CortexM │ cortex-m55    │
╰──────────────┴───────────────────────────┴───────────────╯

In [ ]:
resolved = rt_session.resolve()
config_table = Table(title="Resolved RT Experiment")
config_table.add_column("Setting")
config_table.add_column("Value")
config_table.add_row("Model", str(resolved.model.path))
config_table.add_row("Engine", resolved.engine.type.value)
config_table.add_row("Board", resolved.target.board)
config_table.add_row("Toolchain", resolved.target.toolchain.value)
config_table.add_row("Transport", resolved.target.transport.value)
config_table.add_row("Iterations", str(resolved.profiling.iterations))
config_table.add_row("Warmup", str(resolved.profiling.warmup))
config_table.add_row("PMU groups", ", ".join((resolved.profiling.pmu_counters or {}).keys()))
console.print(config_table)

if rt_result is not None:
    print("Run ID:", rt_result.metadata.run_id)
    print("Available PMU presets:", sorted(rt_result.pmu.presets))
    print("Available compute groups:", sorted(rt_result.pmu.groups))
    print("Firmware metadata:", rt_result.pmu.meta)

## 4. Summarize Runtime and Layer Execution

HPX measures model operators rather than host tasks. The equivalent headline view reports layer count, summed instrumented cycles, clean end-to-end latency, capture duration, overflow status, and operator diversity.

In [ ]:
summary_rows = []
if rt_result is not None:
    firmware = rt_result.pmu.meta
    timing = rt_result.metadata.timing
    summary_rows = [
        ("Layers", rt_result.layer_count),
        ("Operator types", len({layer.op for layer in rt_result.layers})),
        ("Instrumented layer cycles", rt_result.total_cycles),
        ("Clean inference cycles", firmware.clean_infer_avg_cycles),
        ("Clean inference time (us)", firmware.clean_infer_avg_us),
        ("Capture duration (s)", timing.capture_duration_s if timing else None),
        ("Counter overflow", rt_result.overflow_detected),
    ]

summary_table = Table(title="Profile Summary")
summary_table.add_column("Metric")
summary_table.add_column("Value", justify="right")
for label, value in summary_rows:
    summary_table.add_row(label, "-" if value is None else f"{value:,}" if isinstance(value, (int, float)) else str(value))
console.print(summary_table if summary_rows else "[dim]No profile result loaded.[/dim]")

## 5. Explore the HPX Layer and Memory Hierarchy

TFLite operator results are a flat execution sequence, not a parent-child task trace. HPX adds two useful structures: operator-type aggregation across that sequence and a memory plan whose regions contain named consumers such as weights, arena, and code.

In [ ]:
operator_summary = []
if rt_result is not None:
    by_op = {}
    for layer in rt_result.layers:
        bucket = by_op.setdefault(layer.op, {"count": 0, "cycles": 0.0})
        bucket["count"] += 1
        bucket["cycles"] += layer.cycles or 0
    operator_summary = sorted(
        ({"op": op, **values} for op, values in by_op.items()),
        key=lambda row: row["cycles"],
        reverse=True,
    )

op_table = Table(title="Operator Hierarchy Summary")
op_table.add_column("Operator")
op_table.add_column("Layers", justify="right")
op_table.add_column("Total cycles", justify="right")
op_table.add_column("Share", justify="right")
for row in operator_summary:
    share = row["cycles"] / rt_result.total_cycles if rt_result and rt_result.total_cycles else 0
    op_table.add_row(row["op"], str(row["count"]), f"{row['cycles']:,.0f}", f"{share:.1%}")
console.print(op_table if operator_summary else "[dim]No operator data loaded.[/dim]")

memory_plan = rt_result.metadata.memory_plan if rt_result is not None else None
if memory_plan is not None:
    for region in memory_plan.regions:
        print(f"{region.region.value}: {region.used:,}/{region.capacity:,} bytes")
        for consumer in region.consumers:
            print(f"  - {consumer.kind.value}: {consumer.name} ({consumer.size:,} bytes)")
else:
    print("No resolved memory plan is available until a profile completes.")

## 6. Analyze Layer Timing Distributions

Group layers by operator and compute count, total, mean, median, p90, and maximum cycles. This exposes both frequently executed operators and individual timing outliers without introducing a dataframe dependency.

In [ ]:
def percentile(values, fraction):
    if not values:
        return 0.0
    ordered = sorted(values)
    index = min(len(ordered) - 1, round((len(ordered) - 1) * fraction))
    return ordered[index]


timing_rows = []
if rt_result is not None:
    grouped = {}
    for layer in rt_result.layers:
        grouped.setdefault(layer.op, []).append(float(layer.cycles or 0))
    for op, values in grouped.items():
        timing_rows.append(
            {
                "op": op,
                "count": len(values),
                "total": sum(values),
                "mean": mean(values),
                "median": median(values),
                "p90": percentile(values, 0.90),
                "max": max(values),
            }
        )
    timing_rows.sort(key=lambda row: row["total"], reverse=True)

timing_table = Table(title="Layer Timing Distribution")
for label in ("Operator", "Count", "Total", "Mean", "Median", "P90", "Max"):
    timing_table.add_column(label, justify="right" if label != "Operator" else "left")
for row in timing_rows:
    timing_table.add_row(
        row["op"], str(row["count"]),
        f"{row['total']:,.0f}", f"{row['mean']:,.0f}",
        f"{row['median']:,.0f}", f"{row['p90']:,.0f}", f"{row['max']:,.0f}",
    )
console.print(timing_table if timing_rows else "[dim]No layer timings loaded.[/dim]")

## 7. Visualize Execution Across Compute-Unit Lanes

HPX profiles one embedded inference core per run, so it does not expose host worker threads. Its closest lane view is the set of PMU compute groups (`cpu`, `memory`, `mve`) merged across firmware passes. The table below ranks layers within each available lane and supports a layer-index zoom.

In [ ]:
ZOOM_LAYER_RANGE = (0, 20)
lane_table = Table(title=f"Compute Lanes: layers {ZOOM_LAYER_RANGE[0]}..{ZOOM_LAYER_RANGE[1]}")
lane_table.add_column("Lane")
lane_table.add_column("Layer")
lane_table.add_column("Operator")
lane_table.add_column("Leading counter")
lane_table.add_column("Value", justify="right")

if rt_result is not None:
    for group_name, layers in sorted(rt_result.pmu.groups.items()):
        for layer in layers[ZOOM_LAYER_RANGE[0]:ZOOM_LAYER_RANGE[1]]:
            leading = max(layer.counters, key=layer.counters.get) if layer.counters else "cycles"
            value = layer.counters.get(leading, layer.cycles or 0)
            lane_table.add_row(group_name, str(layer.id), layer.op, leading, f"{value:,.0f}")
console.print(lane_table if lane_table.row_count else "[dim]No compute-lane data loaded.[/dim]")

## 8. Identify Imbalances and Bottlenecks

On a single inference core, imbalance means concentration rather than worker skew: a small number of layers consuming most cycles, high stall counters, memory-refill pressure, or a large gap between instrumented and clean inference timing. The following rankings report measured values; interpretation remains workload-specific.

In [ ]:
hotspots = []
if rt_result is not None:
    hotspots = sorted(rt_result.layers, key=lambda layer: layer.cycles or 0, reverse=True)

hotspot_table = Table(title="Cycle Hotspots")
hotspot_table.add_column("Rank", justify="right")
hotspot_table.add_column("Layer")
hotspot_table.add_column("Operator")
hotspot_table.add_column("Cycles", justify="right")
hotspot_table.add_column("Profile share", justify="right")
hotspot_table.add_column("Overflow")
for rank, layer in enumerate(hotspots[:12], 1):
    cycles = layer.cycles or 0
    share = cycles / rt_result.total_cycles if rt_result and rt_result.total_cycles else 0
    hotspot_table.add_row(
        str(rank), str(layer.id), layer.op, f"{cycles:,.0f}", f"{share:.1%}", str(layer.overflow)
    )
console.print(hotspot_table if hotspots else "[dim]No hotspot data loaded.[/dim]")

if hotspots and rt_result.total_cycles:
    top_share = sum((layer.cycles or 0) for layer in hotspots[:5]) / rt_result.total_cycles
    print(f"Measured concentration: top five layers account for {top_share:.1%} of layer cycles.")

## 9. Filter and Drill Down

Compose filters by operator text, layer ID range, minimum cycle count, overflow state, and counter availability. The filtered list can feed the same summary and hotspot views without mutating the original `ProfileResult`.

In [ ]:
def filter_layers(
    layers,
    *,
    op_contains=None,
    id_range=None,
    min_cycles=0,
    overflow_only=False,
    counter=None,
):
    selected = []
    for index, layer in enumerate(layers):
        numeric_id = layer.id if isinstance(layer.id, int) else index
        if op_contains and op_contains.lower() not in layer.op.lower():
            continue
        if id_range and not (id_range[0] <= numeric_id < id_range[1]):
            continue
        if (layer.cycles or 0) < min_cycles:
            continue
        if overflow_only and not layer.overflow:
            continue
        if counter and counter not in layer.counters:
            continue
        selected.append(layer)
    return selected


cycle_floor = (rt_result.total_cycles * 0.01) if rt_result is not None else 0
filtered_layers = filter_layers(
    rt_result.layers if rt_result is not None else [],
    id_range=(0, 30),
    min_cycles=cycle_floor,
    counter="ARM_PMU_CPU_CYCLES",
)

filtered_table = Table(title="Filtered Layers")
filtered_table.add_column("Layer")
filtered_table.add_column("Operator")
filtered_table.add_column("Cycles", justify="right")
for layer in sorted(filtered_layers, key=lambda item: item.cycles or 0, reverse=True):
    filtered_table.add_row(str(layer.id), layer.op, f"{layer.cycles or 0:,.0f}")
console.print(filtered_table if filtered_layers else "[dim]No layers match the current drill-down.[/dim]")

## 10. Correlate Layers with Model Provenance

TFLite flatbuffers generally do not carry source file and line information. HPX instead preserves model name, size, SHA-256, layer/operator IDs, and—when AOT analysis is available—original operator IDs through graph transformations. Missing optional analysis support is reported without stopping the notebook.

In [ ]:
if rt_result is not None and rt_result.metadata.model is not None:
    model_info = rt_result.metadata.model
    print("Model:", model_info.name)
    print("Size: ", f"{model_info.size_bytes:,} bytes")
    print("SHA:  ", model_info.sha256)
else:
    print("Runtime model provenance will be available after profiling.")

rt_analysis = None
aot_analysis = None
try:
    rt_analysis = rt_session.analyze()
    print(
        f"RT graph: {len(rt_analysis.layers)} layers, "
        f"{rt_analysis.total_macs:,} MACs, {rt_analysis.total_ops:,} operations"
    )
except hpx.HpxError as exc:
    print("RT analysis unavailable:", exc)

try:
    aot_analysis = aot_session.analyze()
    mapped = sum(layer.original_id is not None for layer in aot_analysis.layers)
    print(
        f"AOT graph: {len(aot_analysis.layers)} layers, "
        f"{mapped} layers mapped to original operator IDs"
    )
except hpx.HpxError as exc:
    print("AOT analysis unavailable:", exc)

## 11. Compare Execution Regions and Engines

First compare two layer windows from the RT run to quantify local concentration. Then, when hardware profiling is enabled, run the independently derived AOT session and use HPX's typed comparison API to compare complete RT and AOT result sets.

In [ ]:
def summarize_region(name, layers):
    cycles = [float(layer.cycles or 0) for layer in layers]
    return {
        "region": name,
        "layers": len(layers),
        "total_cycles": sum(cycles),
        "mean_cycles": mean(cycles) if cycles else 0,
        "dominant_op": max(
            {layer.op for layer in layers},
            key=lambda op: sum((layer.cycles or 0) for layer in layers if layer.op == op),
        ) if layers else "-",
    }

region_rows = []
if rt_result is not None:
    midpoint = len(rt_result.layers) // 2
    region_rows = [
        summarize_region("first half", rt_result.layers[:midpoint]),
        summarize_region("second half", rt_result.layers[midpoint:]),
    ]

region_table = Table(title="Execution Region Comparison")
for label in ("Region", "Layers", "Total cycles", "Mean cycles", "Dominant operator"):
    region_table.add_column(label)
for row in region_rows:
    region_table.add_row(
        row["region"], str(row["layers"]), f"{row['total_cycles']:,.0f}",
        f"{row['mean_cycles']:,.0f}", row["dominant_op"],
    )
if len(region_rows) == 2:
    delta = region_rows[1]["total_cycles"] - region_rows[0]["total_cycles"]
    region_table.caption = f"Second - first total-cycle delta: {delta:+,.0f}"
console.print(region_table if region_rows else "[dim]No execution regions loaded.[/dim]")

In [ ]:
aot_result = None
comparison = None
if RUN_HARDWARE:
    try:
        aot_result = aot_session.profile()
        comparison = base.compare(
            rt_result,
            aot_result,
            output_dir=COMPARE_RESULTS,
        )
        print(f"Compared {len(comparison.layer_rows)} aligned layers")
        for metric in comparison.metrics:
            if metric.delta is not None:
                print(
                    f"{metric.name}: {metric.baseline} -> {metric.candidate} "
                    f"(delta {metric.delta:+g})"
                )
    except hpx.HpxError as exc:
        print("AOT profile or comparison unavailable:", exc)
else:
    print("RT-vs-AOT hardware comparison skipped.")

## Configuration Snapshots, Overlays, and Optional Power

`Session.from_yaml()` snapshots configuration at construction, so later file edits cannot change an existing experiment. Model Explorer overlays remain ordinary report paths.

Joulescope power capture is a separate opt-in branch. `RUN_HARDWARE` controls ordinary RT/AOT profiling; `RUN_POWER` independently controls the power-enabled run. Leave `RUN_POWER = False` when no Joulescope is connected.

In [ ]:
with tempfile.TemporaryDirectory(prefix="hpx_session_") as temp_dir:
    config_path = Path(temp_dir) / "profile.yml"
    config_path.write_text(
        f"model:\n  path: {MODEL}\ntarget:\n  board: {BOARD}\n"
    )
    yaml_session = hpx.Session.from_yaml(config_path)
    config_path.write_text("model:\n  path: changed-after-snapshot.tflite\n")
    assert yaml_session.resolve().model.path == MODEL
    print("YAML snapshot model:", yaml_session.resolve().model.path)

power_session = (
    rt_session
    .with_power(enabled=True, duration_s=10)
    .with_output(dir=RESULTS_ROOT / "power")
)
print("Power enabled in branch:", power_session.resolve().power.enabled)
print("Power duration:         ", power_session.resolve().power.duration_s)

power_result = None
if RUN_POWER:
    if not RUN_HARDWARE:
        raise RuntimeError("Set RUN_HARDWARE = True before enabling RUN_POWER")
    power_result = power_session.profile()
else:
    print("Power capture skipped. Set RUN_POWER = True only with a Joulescope connected.")

overlay_paths = [
    path
    for result in (rt_result, aot_result)
    if result is not None
    for path in result.report_paths
    if path.name.startswith("me_overlay_")
]
print("Model Explorer overlays:", len(overlay_paths))
for path in overlay_paths[:8]:
    print(" ", path)

## 12. Export Analysis Results and Visualizations

Write derived tables beneath the repository-ignored `results/` tree, never into fixtures. CSV and JSON retain machine-readable analysis; a captured Rich hotspot table provides a portable text visualization. HPX's own reports and Model Explorer overlays remain in each run directory.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
summary_csv = EXPORT_DIR / "operator_timing_summary.csv"
summary_json = EXPORT_DIR / "showcase_summary.json"
hotspot_text = EXPORT_DIR / "cycle_hotspots.txt"

fieldnames = ["op", "count", "total", "mean", "median", "p90", "max"]
with summary_csv.open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(timing_rows)

summary_json.write_text(
    json.dumps(
        {
            "hpx_version": hpx.__version__,
            "model": str(MODEL),
            "board": BOARD,
            "profile_loaded": rt_result is not None,
            "layers": rt_result.layer_count if rt_result is not None else 0,
            "total_cycles": rt_result.total_cycles if rt_result is not None else 0,
            "operator_summary": operator_summary,
            "overlays": [str(path) for path in overlay_paths],
        },
        indent=2,
    ) + "\n"
)

recording_console = Console(record=True, width=110)
recording_console.print(hotspot_table if hotspots else "No hotspot data loaded.")
recording_console.save_text(str(hotspot_text))

for path in (summary_csv, summary_json, hotspot_text):
    print(f"{path.name}: {path.stat().st_size:,} bytes -> {path}")
print(summary_json.read_text()[:500])

## 13. Validate the New Walkthrough

The final cell validates the safe-path objects on every run and tightens its assertions when real profile data is present. Restart the kernel and run all cells before promoting this candidate over the original walkthrough.

In [ ]:
assert MODEL.is_file()
assert MODEL.read_bytes()[4:8] == b"TFL3"
assert doctor.checks
assert engines
assert boards
assert cpu_counters
assert resolved.model.path == MODEL
assert resolved.target.board == BOARD
assert resolved.target.toolchain == hpx.Toolchain.ARM_NONE_EABI_GCC
assert summary_csv.is_file()
assert summary_json.is_file()
assert hotspot_text.is_file()
assert set(fieldnames) == {"op", "count", "total", "mean", "median", "p90", "max"}

if rt_result is not None:
    assert rt_result.layers
    assert rt_result.layer_count > 0
    assert timing_rows
    assert operator_summary
    assert all(hasattr(layer, "counters") for layer in rt_result.layers)

print("Showcase validation passed.")
print("Packaged model:", MODEL)
print("Results directory:", RESULTS_ROOT)